In [1]:
import os

In [2]:
%pwd

'c:\\Users\\Lenovo\\Desktop\\End-to-End-Project\\kidney_disease_classification_end-to-end-DL\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'c:\\Users\\Lenovo\\Desktop\\End-to-End-Project\\kidney_disease_classification_end-to-end-DL'

In [5]:
import dagshub
dagshub.init(repo_owner='kamranshahid56-tech', repo_name='kidney_disease_classification_end-to-end-DL', mlflow=True)

import mlflow
with mlflow.start_run():
  mlflow.log_param('parameter name', 'value')
  mlflow.log_metric('metric name', 1)

Accessing as kamranshahid56-tech

Initialized MLflow to track repo "kamranshahid56-tech/kidney_disease_classification_end-to-end-DL"

Repository kamranshahid56-tech/kidney_disease_classification_end-to-end-DL initialized!

c:\Users\Lenovo\anaconda3\envs\kidney\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🏃 View run nimble-carp-196 at: https://dagshub.com/kamranshahid56-tech/kidney_disease_classification_end-to-end-DL.mlflow/#/experiments/0/runs/ecbffc45b9eb44538f31e8f1de9598e3
🧪 View experiment at: https://dagshub.com/kamranshahid56-tech/kidney_disease_classification_end-to-end-DL.mlflow/#/experiments/0


In [17]:
os.environ["MLFLOW_TRACKING_URI"] = "https://dagshub.com/kamranshahid56-tech/kidney_disease_classification_end-to-end-DL.mlflow"
os.environ["MLFLOW_TRACKING_USERNAME"] = "kamranshahid56-tech"
os.environ["MLFLOW_TRACKING_PASSWORD"] = "c74df82290749ab1b393dd2b15ce6bf7b96330db"

In [7]:
import tensorflow as tf

In [8]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class EvaluationConfig:
    path_of_model: Path
    training_data: Path
    all_params: dict
    mlflow_uri: str
    params_image_size: list
    params_batch_size: int

In [9]:
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml, create_directories, save_json

In [14]:
type(save_json)

ensure.main.WrappedFunction

In [10]:
class ConfigurationManager:
    def __init__(self, 
                 config_file_path = CONFIG_FILE_PATH, 
                 params_file_path = PARAMS_FILE_PATH
                 ):
        self.config = read_yaml(config_file_path)
        self.params = read_yaml(params_file_path)

        create_directories([self.config.artifacts_root])

    def get_evaluation_config(self) -> EvaluationConfig:
        eval_config = EvaluationConfig(
            path_of_model = "artifacts/training/model.h5",
            training_data = "artifacts/data_ingestion/kidney-ct-scan-image",
            all_params=self.params,
            mlflow_uri= "https://dagshub.com/kamranshahid56-tech/kidney_disease_classification_end-to-end-DL.mlflow",
            params_image_size=self.params.IMAGE_SIZE,
            params_batch_size=self.params.BATCH_SIZE
        )
        return eval_config

In [11]:
import tensorflow as tf
import pathlib as Path
import mlflow
import mlflow.keras
from urllib.parse import urlparse

In [15]:
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml, create_directories, save_json

class Evaluation:
    def __init__(self, config: EvaluationConfig):
        self.config = config
    
    def _valid_generator(self):

        datagenerator_kwargs = dict(
            rescale = 1./255,
            validation_split=0.30
        )

        dataflow_kwargs = dict(
            target_size=self.config.params_image_size[:-1],
            batch_size=self.config.params_batch_size,
            interpolation="bilinear"
        )

        valid_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
            **datagenerator_kwargs
        )

        self.valid_generator = valid_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            subset="validation",
            shuffle=False,
            **dataflow_kwargs
        )

    @staticmethod
    def _load_model(path: Path) -> tf.keras.Model:
        return tf.keras.models.load_model(path)
    

    def evaluation(self):
        self.model = self._load_model(self.config.path_of_model)
        self._valid_generator()
        self.score = self.model.evaluate(self.valid_generator)
        self.save_score()

    def save_score(self):
        scores = {"loss": self.score[0], "accuracy": self.score[1]}
        save_json(path=Path("scores.json"), data=scores)

    def log_into_mlflow(self):
        mlflow.set_registry_uri(self.config.mlflow_uri)
        tracking_url_type_store = urlparse(mlflow.get_tracking_uri()).scheme

        with mlflow.start_run():
            mlflow.log_params(self.config.all_params)
            mlflow.log_metrics({"loss": self.score[0], "accuracy": self.score[1]})

            if tracking_url_type_store != "file":
                mlflow.keras.log_model(self.model, "model", registered_model_name="VGG16Model")
            else:
                mlflow.keras.log_model(self.model, "model")

In [16]:
try:
    config = ConfigurationManager()
    eval_config = config.get_evaluation_config()
    evaluation = Evaluation(eval_config)
    evaluation.evaluation()
    evaluation.log_into_mlflow()
except Exception as e:
    raise e

[2026-03-14 01:45:13,621: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-03-14 01:45:13,640: INFO: common: yaml file: params.yaml loaded successfully]
[2026-03-14 01:45:13,649: INFO: common: created directory at: artifacts]


[2026-03-14 01:45:14,349: WARNING: saving_utils: Compiled the loaded model, but the compiled metrics have yet to be built. `model.compile_metrics` will be empty until you train or evaluate the model.]
Found 139 images belonging to 2 classes.
9/9 ━━━━━━━━━━━━━━━━━━━━ 38s 4s/step - accuracy: 0.5036 - loss: 4.2315
[2026-03-14 01:45:52,935: INFO: common: json file saved at: scores.json]


2026/03/14 01:45:55 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/14 01:45:57 WARNING mlflow.keras.save: You are saving a Keras model without specifying model signature.


[2026-03-14 01:47:28,410: WARNING: connectionpool: Retrying (Retry(total=6, connect=7, read=6, redirect=7, status=7)) after connection broken by 'ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host')': /kamranshahid56-tech/kidney_disease_classification_end-to-end-DL.mlflow/api/2.0/mlflow-artifacts/artifacts/33f91dbf54c9425886f0469b3154f0f2/models/m-21cd96b5d9034e31b0cfd2e5e145a50d/artifacts/data/model.keras]


Successfully registered model 'VGG16Model'.
2026/03/14 01:49:46 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: VGG16Model, version 1
Created version '1' of model 'VGG16Model'.


🏃 View run youthful-mule-200 at: https://dagshub.com/kamranshahid56-tech/kidney_disease_classification_end-to-end-DL.mlflow/#/experiments/0/runs/c334fd91f212462691406c385350ba05
🧪 View experiment at: https://dagshub.com/kamranshahid56-tech/kidney_disease_classification_end-to-end-DL.mlflow/#/experiments/0


In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

print("URI:", os.environ.get("MLFLOW_TRACKING_URI"))
print("USER:", os.environ.get("MLFLOW_TRACKING_USERNAME"))
print("PASS:", os.environ.get("MLFLOW_TRACKING_PASSWORD"))

URI: https://dagshub.com/kamranshahid56-tech/kidney_disease_classification_end-to-end-DL.mlflow
USER: kamranshahid56-tech
PASS: c74df82290749ab1b393dd2b15ce6bf7b96330db


In [2]:
import mlflow

# Add this before start_run
if mlflow.active_run():
    mlflow.end_run()  # ← close any hanging run

c:\Users\Lenovo\anaconda3\envs\kidney\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
